# DDoS Attack Detection and Traffic Analysis using Plotly
### Comprehensive Exploratory Data Analysis and Interactive Visualizations

---

## 1. Project Overview and Context

This notebook presents an in-depth exploratory data analysis (EDA) and interactive visualization suite for the **DDoS SDN Network Traffic Dataset** (`ddosdn20258_binary_0n1d.xlsx`).

### Key Objectives:
1. **Understand Class Balance**: Quantify distribution between legitimate traffic (`Benign = 0`) and malicious traffic (`DDoS Attack = 1`).
2. **Protocol Analysis**: Examine protocol-specific traffic distribution (ICMP, TCP, UDP).
3. **Statistical Profile of Features**: Identify distinct behavioral signatures of DDoS floods (e.g. packet rates, flow durations, packet length variance, inter-arrival times).
4. **Interactive Visualization**: Leverage **Plotly** to create interactive dashboards (2D/3D scatter plots, violin/box distributions, correlation heatmaps, radar profiles).
5. **Feature Separability**: Highlight key discriminative attributes that can feed machine learning anomaly detection models.

---
## 2. Environment Setup and Library Imports

We import `pandas` and `numpy` for data manipulation, and `plotly.express`, `plotly.graph_objects`, and `plotly.subplots` for interactive visualizations.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set default Plotly styling
import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries and dependencies successfully imported.")


Libraries and dependencies successfully imported.


---
## 3. Loading and Preprocessing Dataset

Loading the dataset from `ddosdn20258_binary_0n1d.xlsx` and enriching it with human-readable labels:
* **Protocol**: `1 -> ICMP`, `6 -> TCP`, `17 -> UDP`
* **Label**: `0 -> Benign (Normal)`, `1 -> DDoS Attack`

In [2]:
excel_path = "ddosdn20258_binary_0n1d.xlsx"
df = pd.read_excel(excel_path)

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Map categorical representations for clarity
protocol_map = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
label_map = {0: 'Benign', 1: 'DDoS Attack'}

df['Protocol_Name'] = df['Protocol'].map(protocol_map).fillna(df['Protocol'].astype(str))
df['Class_Label'] = df['Label'].map(label_map)

# Quick look at first 5 records
df.head()


Dataset Shape: 200,378 rows x 9 columns


,Protocol,Flow Duration,Flow IAT Max,Bwd Pkts/s,Pkt Len Std,Pkt Len Var,Bwd IAT Tot,Flow Pkts/s,Label,Protocol_Name,Class_Label
0,1,0.000034,0.000016,58823.529412,0.000000,0.00000,0.000000,147058.823529,0,ICMP,Benign
1,1,0.000022,0.000010,90909.090909,0.000000,0.00000,0.000000,227272.727273,0,ICMP,Benign
2,1,123.474345,20.309947,0.000000,244.483432,59772.14876,0.000000,0.178175,0,ICMP,Benign
3,6,98.564779,24.911137,0.101456,0.000000,0.00000,98.564754,0.213058,0,TCP,Benign
4,6,100.873344,22.601968,0.198268,0.000000,0.00000,100.873316,0.406450,0,TCP,Benign


---
## 4. Dataset Overview and Key Metrics

Summary of high-level Key Performance Indicators (KPIs) and data integrity metrics.

In [3]:
total_records = len(df)
ddos_count = int((df['Label'] == 1).sum())
benign_count = int((df['Label'] == 0).sum())
ddos_pct = (ddos_count / total_records) * 100
benign_pct = (benign_count / total_records) * 100

# Create KPI Indicator Cards
fig_kpi = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]]
)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=total_records,
    title={"text": "<b>Total Network Flows</b><br><span style='font-size:12px;color:gray'>Full Dataset Size</span>", "font": {"size": 14}},
    number={"valueformat": ",d", "font": {"color": "#2C3E50", "size": 28}}
), row=1, col=1)

fig_kpi.add_trace(go.Indicator(
    mode="number+delta",
    value=ddos_count,
    title={"text": "<b>DDoS Attack Flows</b><br><span style='font-size:12px;color:gray'>Label = 1</span>", "font": {"size": 14}},
    delta={"reference": total_records / 2, "relative": False, "valueformat": ",d", "position": "bottom"},
    number={"valueformat": ",d", "font": {"color": "#E74C3C", "size": 28}}
), row=1, col=2)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=benign_count,
    title={"text": "<b>Benign Flows</b><br><span style='font-size:12px;color:gray'>Label = 0</span>", "font": {"size": 14}},
    number={"valueformat": ",d", "font": {"color": "#27AE60", "size": 28}}
), row=1, col=3)

fig_kpi.add_trace(go.Indicator(
    mode="number",
    value=ddos_pct,
    number={"suffix": "%", "valueformat": ".1f", "font": {"color": "#8E44AD", "size": 28}},
    title={"text": "<b>Attack Ratio</b><br><span style='font-size:12px;color:gray'>DDoS Concentration</span>", "font": {"size": 14}}
), row=1, col=4)

fig_kpi.update_layout(
    height=180,
    margin=dict(l=20, r=20, t=40, b=20),
    paper_bgcolor="rgba(245, 247, 250, 0.6)",
)

fig_kpi.show()


### Statistical Summary Table by Class
Comparing descriptive metrics (Mean, Standard Deviation, Min, Median, Max) between Benign and DDoS flows.

In [4]:
features = ['Flow Duration', 'Flow IAT Max', 'Bwd Pkts/s', 'Pkt Len Std', 'Pkt Len Var', 'Bwd IAT Tot', 'Flow Pkts/s']

summary_df = df.groupby('Class_Label')[features].agg(['mean', 'std', 'median']).T.reset_index()
summary_df.columns = ['Feature', 'Metric', 'Benign', 'DDoS Attack']
summary_df['Benign'] = summary_df['Benign'].apply(lambda x: f"{x:,.3f}")
summary_df['DDoS Attack'] = summary_df['DDoS Attack'].apply(lambda x: f"{x:,.3f}")

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Feature Name</b>', '<b>Metric</b>', '<b>Benign (Normal)</b>', '<b>DDoS Attack</b>'],
        fill_color='#2C3E50',
        align='left',
        font=dict(color='white', size=13),
        height=32
    ),
    cells=dict(
        values=[summary_df['Feature'], summary_df['Metric'], summary_df['Benign'], summary_df['DDoS Attack']],
        fill_color=[['#F8F9F9', 'white'] * (len(summary_df) // 2 + 1)],
        align=['left', 'center', 'right', 'right'],
        font=dict(color='#2C3E50', size=12),
        height=26
    )
)])

fig_table.update_layout(
    title="<b>Feature Distribution Summary: Benign vs DDoS Traffic</b>",
    height=480,
    margin=dict(l=10, r=10, t=50, b=10)
)
fig_table.show()


---
## 5. Class Balance and Protocol Distribution

Examining:
1. Overall Traffic Composition (Donut Chart).
2. Protocol-wise Breakdown across Benign and DDoS Traffic.

In [5]:
# Subplot: Class Distribution & Protocol Comparison
fig_dist = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>Class Balance (Benign vs DDoS)</b>", "<b>Traffic Volume by Protocol & Class</b>"),
    specs=[[{"type": "pie"}, {"type": "bar"}]]
)

# 1. Donut Chart
class_counts = df['Class_Label'].value_counts()
fig_dist.add_trace(
    go.Pie(
        labels=class_counts.index,
        values=class_counts.values,
        hole=0.5,
        marker=dict(colors=['#E74C3C', '#2ECC71'], line=dict(color='#FFFFFF', width=2)),
        textinfo='label+percent+value',
        textfont=dict(size=12),
        hoverinfo='label+value+percent'
    ),
    row=1, col=1
)

# 2. Grouped Bar Chart by Protocol
proto_class = df.groupby(['Protocol_Name', 'Class_Label']).size().reset_index(name='Count')
benign_proto = proto_class[proto_class['Class_Label'] == 'Benign']
ddos_proto = proto_class[proto_class['Class_Label'] == 'DDoS Attack']

fig_dist.add_trace(
    go.Bar(
        x=benign_proto['Protocol_Name'],
        y=benign_proto['Count'],
        name='Benign',
        marker_color='#2ECC71',
        text=benign_proto['Count'].apply(lambda x: f"{x:,}"),
        textposition='auto'
    ),
    row=1, col=2
)

fig_dist.add_trace(
    go.Bar(
        x=ddos_proto['Protocol_Name'],
        y=ddos_proto['Count'],
        name='DDoS Attack',
        marker_color='#E74C3C',
        text=ddos_proto['Count'].apply(lambda x: f"{x:,}"),
        textposition='auto'
    ),
    row=1, col=2
)

fig_dist.update_layout(
    barmode='group',
    height=480,
    title_text="<b>Traffic Composition and Protocol Distribution</b>",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_dist.show()


---
## 6. Feature Distributions and Box Plots

DDoS attacks flood networks with high-frequency, low-duration flows. Comparing feature distributions on a log-scale across classes.

In [6]:
# Sample 25,000 rows for smooth boxplot visualization
sample_df = df.sample(n=min(25000, len(df)), random_state=42).copy()

fig_box = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "<b>Flow Packets / Sec (Log Scale)</b>",
        "<b>Flow Duration (Log Scale, sec)</b>",
        "<b>Packet Length Std Dev</b>",
        "<b>Backward Packets / Sec (Log Scale)</b>"
    )
)

colors = {'Benign': '#2ECC71', 'DDoS Attack': '#E74C3C'}

# Flow Pkts/s
for cls in ['Benign', 'DDoS Attack']:
    sub = sample_df[sample_df['Class_Label'] == cls]
    fig_box.add_trace(
        go.Box(y=sub['Flow Pkts/s'] + 1, name=cls, marker_color=colors[cls], showlegend=(cls=='Benign')),
        row=1, col=1
    )
    fig_box.add_trace(
        go.Box(y=sub['Flow Duration'] + 1e-6, name=cls, marker_color=colors[cls], showlegend=False),
        row=1, col=2
    )
    fig_box.add_trace(
        go.Box(y=sub['Pkt Len Std'], name=cls, marker_color=colors[cls], showlegend=False),
        row=2, col=1
    )
    fig_box.add_trace(
        go.Box(y=sub['Bwd Pkts/s'] + 1, name=cls, marker_color=colors[cls], showlegend=False),
        row=2, col=2
    )

fig_box.update_yaxes(type="log", row=1, col=1, title_text="Pkts/s")
fig_box.update_yaxes(type="log", row=1, col=2, title_text="Seconds")
fig_box.update_yaxes(row=2, col=1, title_text="Std Dev (Bytes)")
fig_box.update_yaxes(type="log", row=2, col=2, title_text="Bwd Pkts/s")

fig_box.update_layout(
    height=680,
    title_text="<b>Comparative Feature Distributions: Normal vs Attack Traffic</b>",
    boxmode='group'
)

fig_box.show()


---
## 7. Comparative Histogram and Flow Density Analysis

Examining the separation between benign traffic and DDoS attacks along key flow metrics.

In [7]:
fig_hist = make_subplots(
    rows=1, cols=2,
    subplot_titles=("<b>Log10(Flow Packets/s) Distribution</b>", "<b>Log10(Flow Duration + 1e-6) Distribution</b>")
)

sample_df['log_flow_pkts'] = np.log10(sample_df['Flow Pkts/s'].clip(lower=1))
sample_df['log_flow_dur'] = np.log10(sample_df['Flow Duration'].clip(lower=1e-6))

for cls in ['Benign', 'DDoS Attack']:
    sub = sample_df[sample_df['Class_Label'] == cls]
    fig_hist.add_trace(
        go.Histogram(
            x=sub['log_flow_pkts'],
            name=cls,
            marker_color=colors[cls],
            opacity=0.65,
            nbinsx=40
        ),
        row=1, col=1
    )
    fig_hist.add_trace(
        go.Histogram(
            x=sub['log_flow_dur'],
            name=cls,
            marker_color=colors[cls],
            opacity=0.65,
            nbinsx=40,
            showlegend=False
        ),
        row=1, col=2
    )

fig_hist.update_layout(
    barmode='overlay',
    height=420,
    title_text="<b>Density Comparison: Flow Rate and Duration</b>",
    xaxis_title="Log10(Flow Pkts/s)",
    xaxis2_title="Log10(Flow Duration sec)",
    yaxis_title="Frequency",
    yaxis2_title="Frequency"
)

fig_hist.show()


---
## 8. Correlation Analysis Heatmap

Understanding linear relationships between network flow features and attack labels.

In [8]:
num_cols = ['Protocol', 'Flow Duration', 'Flow IAT Max', 'Bwd Pkts/s', 'Pkt Len Std', 'Pkt Len Var', 'Bwd IAT Tot', 'Flow Pkts/s', 'Label']
corr_matrix = df[num_cols].corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="<b>Correlation Heatmap of Network Traffic Attributes</b>"
)

fig_corr.update_layout(
    height=550,
    xaxis_tickangle=-35,
    margin=dict(l=50, r=50, t=60, b=80)
)

fig_corr.show()


---
## 9. Multi-Dimensional Feature Space and Cluster Separation

Visualizing flow clusters in 2D and 3D feature spaces to observe separation between DDoS attacks and legitimate network activity.

In [9]:
# 2D Scatter Plot: Flow Duration vs Flow Packets/s
sample_plot = df.sample(n=min(10000, len(df)), random_state=42).copy()
sample_plot['Flow Duration (s)'] = sample_plot['Flow Duration'].clip(lower=1e-6)
sample_plot['Flow Pkts/s (clipped)'] = sample_plot['Flow Pkts/s'].clip(lower=1)

fig_scatter2d = px.scatter(
    sample_plot,
    x='Flow Duration (s)',
    y='Flow Pkts/s (clipped)',
    color='Class_Label',
    color_discrete_map={'Benign': '#2ECC71', 'DDoS Attack': '#E74C3C'},
    hover_data=['Protocol_Name', 'Pkt Len Std', 'Flow IAT Max'],
    title="<b>2D Feature Separation: Flow Duration vs Flow Packets/sec</b>",
    log_x=True,
    log_y=True,
    opacity=0.65
)

fig_scatter2d.update_layout(
    height=520,
    legend_title_text="Traffic Class",
    xaxis_title="Flow Duration (Seconds, Log Scale)",
    yaxis_title="Flow Packets/sec (Log Scale)"
)

fig_scatter2d.show()


In [10]:
# 3D Scatter: Flow Duration x Flow Pkts/s x Pkt Len Std
sample_3d = sample_plot.sample(n=min(3000, len(sample_plot)), random_state=42).copy()

fig_scatter3d = px.scatter_3d(
    sample_3d,
    x='Flow Duration (s)',
    y='Flow Pkts/s (clipped)',
    z='Pkt Len Std',
    color='Class_Label',
    color_discrete_map={'Benign': '#2ECC71', 'DDoS Attack': '#E74C3C'},
    log_x=True,
    log_y=True,
    opacity=0.7,
    hover_data=['Protocol_Name', 'Bwd Pkts/s'],
    title="<b>3D Manifold: Duration vs Packet Rate vs Packet Size Variance</b>"
)

fig_scatter3d.update_layout(
    height=650,
    scene=dict(
        xaxis_title='Flow Duration (s)',
        yaxis_title='Flow Pkts/s',
        zaxis_title='Pkt Len Std'
    ),
    margin=dict(l=10, r=10, t=40, b=10)
)

fig_scatter3d.show()


---
## 10. DDoS Behavioral Fingerprint (Radar Chart)

Normalizing mean feature values using Min-Max scaling to trace distinct network fingerprints for benign vs malicious flows.

In [11]:
radar_features = ['Flow Duration', 'Flow IAT Max', 'Bwd Pkts/s', 'Pkt Len Std', 'Bwd IAT Tot', 'Flow Pkts/s']

# Min-Max scale means
scaled_means = {}
for feat in radar_features:
    f_min = df[feat].min()
    f_max = df[feat].max()
    denom = (f_max - f_min) if (f_max - f_min) != 0 else 1
    scaled_means[feat] = df.groupby('Class_Label')[feat].mean().apply(lambda x: (x - f_min) / denom)

radar_df = pd.DataFrame(scaled_means).T

fig_radar = go.Figure()

fig_radar.add_trace(go.Scatterpolar(
    r=radar_df['Benign'].tolist() + [radar_df['Benign'].iloc[0]],
    theta=radar_features + [radar_features[0]],
    fill='toself',
    name='Benign Traffic',
    line_color='#2ECC71',
    fillcolor='rgba(46, 204, 113, 0.25)'
))

fig_radar.add_trace(go.Scatterpolar(
    r=radar_df['DDoS Attack'].tolist() + [radar_df['DDoS Attack'].iloc[0]],
    theta=radar_features + [radar_features[0]],
    fill='toself',
    name='DDoS Attack',
    line_color='#E74C3C',
    fillcolor='rgba(231, 76, 60, 0.25)'
))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, max(radar_df.max().max() * 1.1, 0.1)])
    ),
    title="<b>Behavioral Profile Fingerprint: Benign vs DDoS Attacks</b>",
    height=520,
    showlegend=True
)

fig_radar.show()


---
## 11. Protocol-Specific Flood Characteristics

Analyzing how attack dynamics vary between ICMP, TCP, and UDP protocols.

In [12]:
proto_summary = df.groupby(['Protocol_Name', 'Class_Label'])[['Flow Pkts/s', 'Flow Duration', 'Pkt Len Std']].mean().reset_index()

fig_proto = make_subplots(
    rows=1, cols=3,
    subplot_titles=("<b>Mean Flow Packets/s</b>", "<b>Mean Flow Duration (s)</b>", "<b>Mean Packet Length Std Dev</b>")
)

for i, metric in enumerate(['Flow Pkts/s', 'Flow Duration', 'Pkt Len Std'], 1):
    for cls in ['Benign', 'DDoS Attack']:
        sub = proto_summary[proto_summary['Class_Label'] == cls]
        fig_proto.add_trace(
            go.Bar(
                x=sub['Protocol_Name'],
                y=sub[metric],
                name=cls,
                marker_color=colors[cls],
                showlegend=(i == 1)
            ),
            row=1, col=i
        )

fig_proto.update_layout(
    barmode='group',
    height=420,
    title_text="<b>Attack Signatures Across Protocols (ICMP vs TCP vs UDP)</b>"
)

fig_proto.show()


---
## 12. Feature Discriminative Power (Separation Score)

Quantifying the separation power of each feature using Normalized Fisher Criterion / Mean Separation Index:
$$\text{Separation Score} = \frac{|\mu_{\text{DDoS}} - \mu_{\text{Benign}}|}{\sqrt{\sigma_{\text{DDoS}}^2 + \sigma_{\text{Benign}}^2}}$$

In [13]:
eval_features = ['Flow Pkts/s', 'Bwd Pkts/s', 'Flow Duration', 'Flow IAT Max', 'Pkt Len Std', 'Pkt Len Var', 'Bwd IAT Tot']

scores = {}
for feat in eval_features:
    m_benign = df[df['Label'] == 0][feat].mean()
    m_ddos = df[df['Label'] == 1][feat].mean()
    v_benign = df[df['Label'] == 0][feat].var()
    v_ddos = df[df['Label'] == 1][feat].var()
    
    score = abs(m_ddos - m_benign) / np.sqrt(v_benign + v_ddos + 1e-9)
    scores[feat] = score

sep_df = pd.DataFrame(list(scores.items()), columns=['Feature', 'Separation_Score']).sort_values('Separation_Score', ascending=True)

fig_sep = px.bar(
    sep_df,
    x='Separation_Score',
    y='Feature',
    orientation='h',
    color='Separation_Score',
    color_continuous_scale='Viridis',
    title="<b>Feature Separability Power for DDoS Anomaly Detection</b>",
    text=sep_df['Separation_Score'].apply(lambda x: f"{x:.3f}")
)

fig_sep.update_layout(
    height=420,
    xaxis_title="Fisher Discriminant Separation Score (Higher = Stronger Indicator)",
    yaxis_title="Feature"
)

fig_sep.show()


---
## 13. Key Analytical Insights and Summary

Based on our exploratory data analysis:

1. **Ultra-High Packet Rate (`Flow Pkts/s`)**: DDoS attacks exhibit an average packet rate of **~739,611 pkts/s** compared to **~85,056 pkts/s** for benign flows (almost **9x higher**).
2. **Extremely Short Flow Durations (`Flow Duration`)**: Attack flows last only a fraction of a millisecond on average (**0.0036s**) versus legitimate sessions (**7.49s**), indicative of bursty flood vectors.
3. **Rigid Packet Size Variance (`Pkt Len Std` and `Pkt Len Var`)**: Legitimate traffic exhibits high packet length variation (**std = 113.68 bytes**), whereas DDoS attacks employ fixed-payload packets (**std = 0.007 bytes**).
4. **Inter-Arrival Time Collapse (`Flow IAT Max` and `Bwd IAT Tot`)**: Inter-arrival time is nearly **0** during DDoS floods, reflecting continuous line-rate packet injection.

---
### Ready for Machine Learning Pipeline
These findings confirm that **`Flow Pkts/s`**, **`Flow Duration`**, and **`Pkt Len Std`** provide sharp boundaries for lightweight thresholding, Random Forests, XGBoost, or Deep Learning classifiers to detect DDoS attacks in real-time SDN environments.